# 05 - Live annotated view (T6, issue #7)

Visible proof of the tested pipeline: Person enclosed in a colored frame,
live posture score in one corner on every movement. Box color follows the
dwell machine (green while not-bad, red once bad_posture fires, back on
recovered) so the box always agrees with the emitted events.

Base layer is Ultralytics results[0].plot() (boxes plus skeleton, no new
dependency); our overlay adds dwell color plus score text. No new decision
logic here: box_color/annotate_frame are pure and pytest-covered, the loop
reuses 00-03 namespaces.

Manual run (camera + weights required): set `BALDUR_LIVE=1` and run all
cells; press `q` in the window to quit. Display-only: annotated frames
never touch disk (ADR-0002, machine-checked by the persistence scan test).
Limits: CPU inference is single-digit FPS; single tracked Person (largest);
local GUI session required (headless machines keep the JSON-lines events).

In [ ]:
import cv2

UPRIGHT_COLOR = (0, 255, 0)
BAD_COLOR = (0, 0, 255)


def box_color(in_bad):
    """Dwell-state color: red once bad_posture fired, else green."""
    return BAD_COLOR if in_bad else UPRIGHT_COLOR


def annotate_frame(frame, box, score, in_bad):
    """Overlay highlight box plus corner score; returns a new frame."""
    view = frame.copy()
    color = box_color(in_bad)
    if box is not None:
        x1, y1, x2, y2 = (int(value) for value in box)
        cv2.rectangle(view, (x1, y1), (x2, y2), color, 2)
    label = f"score: {score:.0f}" if score is not None else "score: --"
    cv2.putText(view, label, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
    return view

In [ ]:
import json as _json5
from pathlib import Path as _Path5


def _sibling_notebook_path5(name):
    for _base in (_Path5.cwd(), _Path5.cwd() / "notebooks"):
        _candidate = _base / name
        if _candidate.exists():
            return _candidate
    raise FileNotFoundError(name)


def _load_ns5(name):
    """Exec importable cells of a sibling notebook (skips LIVE/DEMO)."""
    with open(_sibling_notebook_path5(name), encoding="utf-8") as _handle:
        _notebook = _json5.load(_handle)
    _namespace = {}
    for _cell in _notebook["cells"]:
        if _cell["cell_type"] != "code":
            continue
        _source = "".join(_cell["source"])
        if "# DEMO" in _source or "# LIVE" in _source:
            continue
        exec(compile(_source, name, "exec"), _namespace)
    return _namespace

In [ ]:
# LIVE: annotated viewer. Manual run with BALDUR_LIVE=1; q quits.
import os as _os5
import time as _time5

if _os5.getenv("BALDUR_LIVE") == "1":
    _ns5 = {
        _name: _load_ns5(_name)
        for _name in (
            "00_env_camera.ipynb",
            "01_pose_score.ipynb",
            "02_events_jsonl.ipynb",
            "03_yolo_inference.ipynb",
        )
    }
    _cam5 = _ns5["00_env_camera.ipynb"]
    try:
        _cap5 = _cam5["open_camera"]()
    except Exception as _open5:
        _cam5["emit_error"](
            getattr(_open5, "code", "CAMERA_UNAVAILABLE"),
            str(_open5) or type(_open5).__name__,
        )
        _cap5 = None
    if _cap5 is not None:
        _sc5 = _ns5["01_pose_score.ipynb"]
        _ev5 = _ns5["02_events_jsonl.ipynb"]
        _inf5 = _ns5["03_yolo_inference.ipynb"]
        try:
            _model5, _ = _inf5["load_pose_model"]()
            _state5 = _ev5["new_state"]()
            _warmed5 = False
            try:
                _wframe5 = _cam5["read_frame"](_cap5)
                _inf5["infer_frame"](_model5, _wframe5)
                _warmed5 = True
            except Exception as _wexc5:
                _cam5["emit_error"](
                    getattr(_wexc5, "code", "FRAME_READ_FAILED"),
                    str(_wexc5) or type(_wexc5).__name__,
                )
            _prev5 = _time5.time()
            while _warmed5:
                try:
                    _frame5 = _cam5["read_frame"](_cap5)
                except Exception as _read5:
                    _cam5["emit_error"](
                        getattr(_read5, "code", "FRAME_READ_FAILED"),
                        str(_read5) or type(_read5).__name__,
                    )
                    break
                _now5 = _time5.time()
                _dt5, _prev5 = _now5 - _prev5, _now5
                _h5, _w5 = _frame5.shape[0], _frame5.shape[1]
                try:
                    _results5 = _model5(_frame5, verbose=False)
                except Exception as _infer5exc:
                    _ev5["emit_event"](
                        {
                            "event": "error",
                            "code": getattr(_infer5exc, "code", "INFERENCE_FAILED"),
                            "message": str(_infer5exc) or type(_infer5exc).__name__,
                        }
                    )
                    break
                _view5 = _results5[0].plot() if _results5 else _frame5
                _persons5 = _inf5["extract_persons"](_results5[0]) if _results5 else []
                if not _persons5:
                    _score5, _box5 = None, None
                else:
                    _p5 = _sc5["select_largest_person"](_persons5)
                    _score5 = _sc5["posture_score"](
                        _p5["xyn"], _p5["visibility"], aspect=_w5 / _h5
                    )
                    _seen5 = _p5["visibility"] >= 0.5
                    if _seen5.any():
                        _xs5 = _p5["xyn"][_seen5, 0] * _w5
                        _ys5 = _p5["xyn"][_seen5, 1] * _h5
                        _box5 = (
                            int(_xs5.min()),
                            int(_ys5.min()),
                            int(_xs5.max()),
                            int(_ys5.max()),
                        )
                    else:
                        _box5 = None
                for _event5 in _ev5["update"](_state5, _score5, _dt5):
                    _ev5["emit_event"](_event5)
                _shown5 = annotate_frame(_view5, _box5, _score5, _state5["in_bad"])
                cv2.imshow("Baldur posture", _shown5)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    break
        finally:
            _cam5["release_camera"](_cap5)
            cv2.destroyAllWindows()
    print("viewer finished")